In [0]:
%run ./silver_common

In [0]:
from pyspark.sql import functions as F
from datetime import date

SOURCE_NAME = "cash"
BRONZE_SOURCE_NAME = "external_cash"
business_date_str = date.today().isoformat()

In [0]:
bronze_df = read_bronze(spark, BRONZE_SOURCE_NAME)
print(f"Bronze row count: {bronze_df.count()}")
bronze_df.printSchema()

Bronze row count: 10
root
 |-- cash_id: string (nullable = true)
 |-- fund_id: string (nullable = true)
 |-- currency: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- source_system: string (nullable = true)
 |-- status: string (nullable = true)
 |-- _run_id: string (nullable = true)
 |-- _source_file_record_count_mismatch: string (nullable = true)
 |-- _ingested_at: timestamp (nullable = true)



### 1. Type casting

In [0]:
typed_df = (
    bronze_df
    .withColumn("cash_id", F.trim(F.col("cash_id")))
    .withColumn("fund_id", F.trim(F.col("fund_id")))            # external code, e.g. FND001
    .withColumn("currency", F.upper(F.trim(F.col("currency"))))
    .withColumn("amount", F.col("amount").cast("double"))
    .withColumn("timestamp", F.to_timestamp("timestamp"))
    .withColumn("status", F.upper(F.trim(F.col("status"))))
    # Bronze doesn't have a business_date column - derive it from timestamp
    # (same fix as 07_silver_external_position - confirm against your real
    # Bronze schema if this still errors).
    .withColumn("business_date", F.to_date(F.col("timestamp")))
)

### 2. Crosswalk join

In [0]:
fund_xwalk = read_crosswalk(spark, "fund")


FUND_XWALK_EXTERNAL_COL = "external_fund_id"
FUND_XWALK_INTERNAL_COL = "internal_fund_id"

fund_xwalk_slim = fund_xwalk.select(
    F.col(FUND_XWALK_EXTERNAL_COL).alias("fund_id"),
    F.col(FUND_XWALK_INTERNAL_COL).alias("internal_fund_id"),
)

joined_df = typed_df.join(fund_xwalk_slim, on="fund_id", how="left")

xwalk_miss_df = joined_df.filter(F.col("internal_fund_id").isNull()) \
    .withColumn("reason_code", F.lit("UNKNOWN_CROSSWALK_MAPPING"))
xwalk_miss_count = xwalk_miss_df.count()
if xwalk_miss_count > 0:
    write_quarantine(xwalk_miss_df, SOURCE_NAME)
    print(f"WARNING: {xwalk_miss_count} rows failed crosswalk lookup - investigate.")

crosswalked_df = joined_df.filter(F.col("internal_fund_id").isNotNull())

### 3. Late-record flag
Cutoff defaults to 12:00 (normal batch is ~09:30-11:00, the seeded
late record is 21:45 - comfortably separated).

In [0]:
late_flagged_df = flag_late(crosswalked_df, ts_col="timestamp", business_date_col="business_date", cutoff_hour=12)
late_count = late_flagged_df.filter(F.col("is_late")).count()
print(f"Late-flagged records: {late_count} (expect 1 - the Day-3 CASH20260917LATE record)")

Late-flagged records: 1 (expect 1 - the Day-3 CASH20260917LATE record)


### 4. Duplicate check - key = cash_id (no break logic needed for cash)

In [0]:
KEY_COLS = ["cash_id"]
COMPARE_COLS = ["internal_fund_id", "amount", "status"]

deduped_df, duplicates_df, breaks_df = split_duplicates(late_flagged_df, KEY_COLS, COMPARE_COLS)
dup_count = duplicates_df.count()
break_count = breaks_df.count()

if dup_count > 0:
    write_quarantine(duplicates_df, SOURCE_NAME)
if break_count > 0:
    write_quarantine(breaks_df.withColumn("reason_code", F.lit("CASH_ATTRIBUTE_BREAK")), SOURCE_NAME)

### 5. Write to Silver
LATE records are loaded, not dropped - flagged so downstream
reconciliation can decide how to treat them.

In [0]:
write_silver(deduped_df, SOURCE_NAME)
print(f"Silver row count: {deduped_df.count()}")

Silver row count: 10


In [0]:
log_dq(spark, SOURCE_NAME, business_date_str, "crosswalk_miss", joined_df.count(), xwalk_miss_count, "UNKNOWN_CROSSWALK_MAPPING")
log_dq(spark, SOURCE_NAME, business_date_str, "duplicate_record", crosswalked_df.count(), dup_count, "DUPLICATE_RECORD")
log_dq(spark, SOURCE_NAME, business_date_str, "late_record", crosswalked_df.count(), late_count, "LATE_ARRIVAL")

/home/spark-e4672acc-bfaf-485c-ac39-c7/.ipykernel/71/command-5696143635338729-2886423099:181: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "created_at": datetime.utcnow().isoformat(),


In [0]:
bronze_count = bronze_df.count()
silver_count = deduped_df.count()
quarantined_count = xwalk_miss_count + dup_count
assert bronze_count == silver_count + quarantined_count, (
    f"Row count mismatch: bronze={bronze_count}, silver={silver_count}, quarantined={quarantined_count}"
)
print(f"OK: bronze={bronze_count} = silver={silver_count} + quarantined={quarantined_count}")

OK: bronze=10 = silver=10 + quarantined=0
